# Entrenamiento de modelo y Búsqueda de Hiperparámetros


En esta sección, utilizaremos el modelo **YOLO (You Only Look Once)**, específicamente 3 variantes pre-entrenadas `yolo26n.pt`, `yolo26s.pt` y `yolo26m.pt`, que se basa en la arquitectura de YOLO. YOLO es un modelo de detección de objetos conocido por su velocidad y precisión. la diferencia entre entros es la cantidad de parametros usados.

**¿Cómo funciona YOLO?**
A diferencia de los detectores de objetos de dos etapas (como Faster R-CNN) que primero proponen regiones y luego las clasifican, YOLO es un detector de una sola etapa. Esto significa que predice las cajas delimitadoras y las probabilidades de clase directamente desde una sola pasada a través de la red neuronal.

Las características clave de YOLO incluyen:
*   **Arquitectura Backbone eficiente**: Utiliza una red de extracción de características optimizada para un rendimiento rápido.
*   **Cabezal de detección desacoplado**: Separa las tareas de clasificación y regresión de cajas delimitadoras, lo que puede mejorar la precisión.
*   **Anchor-free**: Predice directamente los centros y las dimensiones de las cajas, simplificando el proceso y mejorando la generalización.
*   **Pérdida de función avanzada**: Incorpora funciones de pérdida que mejoran la convergencia y la precisión de la detección

Para evaluar el rendimiento de nuestro modelo de detección de objetos, utilizamos varias métricas estándar que nos permiten entender tanto la precisión de las detecciones como la capacidad del modelo para encontrar todos los objetos relevantes.

Las métricas clave que se reportan son:

*   **Precision (P)**: Mide la proporción de detecciones positivas que fueron correctas. Es decir, de todas las cajas que el modelo predijo como objetos, cuántas realmente lo eran.
    *   `P = Verdaderos Positivos / (Verdaderos Positivos + Falsos Positivos)`

*   **Recall (R)**: Mide la proporción de objetos reales que el modelo fue capaz de detectar. Es decir, de todos los objetos que realmente existen en la imagen, cuántos fueron correctamente identificados por el modelo.
    *   `R = Verdaderos Positivos / (Verdaderos Positivos + Falsos Negativos)`

*   **mAP (mean Average Precision)**: Es la métrica más importante y completa en detección de objetos. Representa el promedio de la Precisión Promedio (AP) sobre todas las clases de objetos. La AP es el área bajo la curva de Precision-Recall para una clase específica.

    *   **IoU (Intersection over Union)**: Para que una detección se considere "correcta" (Verdadero Positivo), la superposición entre la caja delimitadora predicha y la caja delimitadora real (ground truth) debe ser mayor que un umbral determinado. Este umbral se mide con el IoU.
        *   `IoU = Área de Intersección / Área de Unión`

    *   **mAP50 (o mAP@0.5)**: Es el mAP calculado utilizando un umbral de IoU de 0.5. Esto significa que una detección se considera correcta si su IoU con la caja real es al menos del 50%. Es una métrica común para evaluaciones rápidas.

    *   **mAP50-95 (o mAP@.5:.95)**: Es el mAP promedio sobre un rango de umbrales de IoU, desde 0.5 hasta 0.95, con pasos de 0.05. Esta métrica es más robusta y exigente, ya que requiere que el modelo sea preciso en la localización de las cajas delimitadoras en diferentes niveles de superposición.

Estas métricas se calculan tanto para el conjunto de validación (durante el entrenamiento) como para el conjunto de prueba (evaluación final) para obtener una visión completa del rendimiento del modelo y detectar posibles sobreajustes.


Una de las grandes ventajas de utilizar modelos de redes neuronales profundas como YOLOv8 es que **no es necesaria una etapa de extracción de características manual o predefinida**.

Tradicionalmente, en el aprendizaje automático clásico, los ingenieros de

In [ ]:
!pip install -r requirements.txt

In [ ]:
from ultralytics import YOLO
import torch

In [ ]:
# Verificación de GPU
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Device to use: {'GPU' if device == '0' else 'CPU'}")

Device to use: GPU


### Parámetros de Entrenamiento

La función `model.train()` permite configurar el proceso de entrenamiento a través de diversos parámetros. A continuación, se explican los más relevantes utilizados en este notebook:

*   **`data`**: Ruta al archivo YAML que define el conjunto de datos (rutas a imágenes de entrenamiento, validación y test, y nombres de clases).
*   **`epochs`**: Número total de épocas para entrenar el modelo. Una época es un ciclo completo a través de todo el conjunto de datos de entrenamiento.
*   **`patience`**: Número de épocas sin mejora en la métrica de validación antes de detener el entrenamiento anticipadamente (Early Stopping). Ayuda a prevenir el sobreajuste.
*   **`batch`**: Tamaño del lote (batch size). Número de imágenes procesadas antes de actualizar los pesos del modelo. Un `batch` más grande puede acelerar el entrenamiento, pero requiere más memoria.
*   **`workers`**: Número de subprocesos (workers) a utilizar para la carga de datos. Un valor más alto puede acelerar la carga de datos, especialmente con conjuntos de datos grandes.
*   **`imgsz`**: Tamaño de la imagen de entrada para el modelo (por ejemplo, 640 para 640x640 píxeles). Las imágenes se redimensionarán a este tamaño antes de ser alimentadas al modelo.
*   **`device`**: Especifica el dispositivo a utilizar para el entrenamiento. Puede ser `'cpu'` o el ID de la GPU (por ejemplo, `'0'` para la primera GPU).
*   **`optimizer`**: Algoritmo de optimización utilizado para ajustar los pesos del modelo. `'auto'` permite a Ultralytics seleccionar el optimizador más adecuado (generalmente AdamW).
*   **`dropout`**: Técnica de regularización donde se "apagan" aleatoriamente un porcentaje de neuronas durante el entrenamiento. Ayuda a prevenir el sobreajuste.
*   **`project`**: Nombre del directorio principal donde se guardarán los resultados del entrenamiento (por ejemplo, `runs/train`).
*   **`name`**: Nombre específico de esta ejecución de entrenamiento, que se creará como un subdirectorio dentro de `project` (por ejemplo, `yolo_vertex_exp`).

In [ ]:
model26n = YOLO('yolo26n.pt')

In [ ]:
data_yaml_path = '../../data/processed/data.yml'

# Entrenamiento con ajustes de hiperparámetros iniciales
results = model26n.train(
    data=data_yaml_path,
    epochs=70,             
    time=None,             
    patience=8,           
    batch=8,               
    workers=2,             
    imgsz=640,              
    device=device,          
    dropout=0.0,           
    project='runs/train/26n', 
    name='yolo_hyper_exp_1',
)

Ultralytics 8.4.55 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../data/processed/data.yml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_hyper_exp_1-10, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

Visualización del archivo `results.png` generado durante el experimento de hiperparámetros.

![Resultados yolo26n](runs/train/26n/yolo_hyper_exp_1-10/results.png)

Observando las curvas de pérdida (train/val) y las métricas de precisión y *recall* a lo largo de las épocas, podemos evaluar el aprendizaje del modelo nano (`yolo26n`). La rápida caída inicial de la pérdida indica que el modelo logra captar las características tempranas. Sin embargo, será crucial validar con la matriz de confusión si tiene algún sesgo hacia una clase determinada, dado que siendo un modelo ligero podría costarle extraer características más finas de objetos menos representados.

#### Matriz de Confusión Normalizada
A continuación se muestra la matriz de confusión normalizada para apreciar qué clases se están prediciendo de manera correcta y en cuáles existe mayor confusión.

![Matriz de Confusión Normalizada yolo26n](runs/train/26n/yolo_hyper_exp_1-10/confusion_matrix_normalized.png)

Al observar la matriz de confusión normalizada (diagonal principal), se puede apreciar el porcentaje de aciertos de cada clase. 
Aquellas celdas en la diagonal con valores superiores a 0.8 indican un rendimiento robusto. Por el contrario, los valores residuales esparcidos fuera de la diagonal nos señalan con qué clases específicas el modelo está confundiendo ciertas predicciones. Si existen confusiones recurrentes o falsos negativos (predicciones hacia el fondo o "Background"), esto confirma la limitación anticipada del modelo *nano* a la hora de separar características complejas.

In [2]:
import pandas as pd

# Lectura del archivo results.csv
results_csv_path = 'runs/train/26n/yolo_hyper_exp_1-10/results.csv'
df_results = pd.read_csv(results_csv_path)

# Limpiar nombres de columnas (YOLO a veces añade espacios en blanco)
df_results.columns = df_results.columns.str.strip()

# Mostrar las últimas 5 filas/épocas para ver cómo terminó el modelo
display(df_results.tail())

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
65,66,4009.25,0.73153,0.61003,0.00321,0.91915,0.87435,0.95437,0.80174,0.63490,0.42415,0.00291,0.000029,0.000029,0.000029
66,67,4060.39,0.72990,0.60603,0.00319,0.91815,0.87582,0.95487,0.80381,0.63039,0.42171,0.00289,0.000024,0.000024,0.000024
67,68,4109.29,0.73423,0.61002,0.00317,0.91796,0.87650,0.95486,0.80353,0.63071,0.42217,0.00289,0.000019,0.000019,0.000019
68,69,4161.66,0.72925,0.60716,0.00315,0.91925,0.87641,0.95479,0.80279,0.63429,0.42325,0.00290,0.000014,0.000014,0.000014
69,70,4211.21,0.73012,0.60681,0.00318,0.92215,0.87280,0.95507,0.80389,0.63121,0.42091,0.00289,0.000009,0.000009,0.000009


#### Análisis de Resultados y Métricas (mAP50-95)
Analizando las últimas épocas del DataFrame resultante, observamos que el modelo alcanzó una **precisión (Precision) aproximada de 92.2%** y un **recall de 87.2%**. A nivel de métricas agregadas globales, el `mAP50` se sitúa en un excelente **95.5%**, lo que indica que detecta de manera muy acertada los objetos cuando el solapamiento (IoU) requerido es superior a 0.5.

Considerando los requerimientos del proyecto, donde nos interesa un rendimiento alto en el **mAP50-95** (evaluando umbrales más estrictos del 50% al 95%), el modelo `yolo26n` ha obtenido aproximadamente un **80.4%**. Esto demuestra una muy buena precisión espacial en las cajas delimitadoras (*bounding boxes*), siendo bastante exacto en ajustar las dimensiones de las predicciones a los objetos reales.

---

In [ ]:
model = YOLO("yolov8s.pt")

results = model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=1024,
    batch=8,
    workers=2,
    cache=True,
    patience=10,
    device=0,
    project="chromosome_detection",
    name="exp_10_epochs"
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp_10_epochs-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective

### Visualización de Resultados yolo26s
Visualización del archivo `results.png` generado durante el experimento de hiperparámetros.

![Resultados yolo26m](\runs\train\26s\exp_10_epochs-2\results.png)

A diferencia del modelo *nano*, el modelo *small* (`yolo26s`) tiene una mayor capacidad de aprendizaje. Observando las curvas de pérdida y métricas a lo largo de las épocas, esperamos ver una mejor capacidad para extraer características finas y una estabilización superior en el rendimiento de validación, considerando que la resolución esperada es mayor

#### Matriz de Confusión Normalizada
![Matriz de Confusión Normalizada yolo26m](\runs\train\26s\exp_10_epochs-2\confusion_matrix_normalized.png)

Al observar detalladamente esta matriz de confusión y compararla con el modelo base, esperamos encontrar una diagonal principal con valores más cercanos a 1.0. Gracias a la mayor complejidad topológica de la variante "small", la incidencia de falsos negativos hacia la clase de "fondo" (Background) debería reducirse de forma evidente, demostrando que al modelo `yolo26s` le cuesta menos distinguir aquellas texturas o variaciones de luz complejas que típicamente confundían al modelo *nano*.

In [ ]:

print(results.results_dict)

{'metrics/precision(B)': 0.9844958468512927, 'metrics/recall(B)': 0.9807661504485313, 'metrics/mAP50(B)': 0.9923342765206903, 'metrics/mAP50-95(B)': 0.8584787351162154, 'fitness': 0.8584787351162154}


#### Análisis de Resultados y Métricas (mAP50-95) para yolo26s
Al evaluar los resultados finales del modelo, confirmamos un rendimiento sobresaliente que justifica plenamente su mayor número de parámetros y la mayor resolución. Las métricas obtenidas son concluyentes:

- **Precisión (Precision):** 98.4%
- **Recall:** 98.07%
- **mAP50:** 99.23%
- **mAP50-95:** 85.84%

En comparación al modelo *nano*, la configuración *small* saltó a un **85.84% en el mAP50-95** (el criterio más estricto del proyecto). A su vez, el Recall se incrementó notablemente hasta un 98.7%, indicando que las omisiones de detecciones (falsos negativos) se redujeron drásticamente. 

La arquitectura más profunda nos ha permitido clasificar de forma robusta e identificar objetos con un margen de precisión espacial altísimo, demostrando que en el contexto de este proyecto (donde nos interesa fundamentalmente el rendimiento en el mAP50-95), la variante "s" de YOLO resulta la mejor opción, compensando significativamente su peso computacional extra.

---
Ahora entrenaremos el modelo medium de modelos YOLO26 manteniendo la resolución inicial de 640

In [ ]:
model26m = YOLO('yolo26m.pt')

In [ ]:
data_yaml_path = '/content/data/processed/data.yml'

# Entrenamiento con ajustes de hiperparámetros iniciales
results = model26m.train(
    data=data_yaml_path,
    epochs=70,
    patience=8,
    batch=32,
    workers=8,
    imgsz=640,
    device=device,
    momentum=0.937,
    weight_decay=0.0005,
    dropout=0.1,
    project='runs/train',
    name='yolo_vertex_exp',
)

Ultralytics 8.4.55 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/processed/data.yml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_vertex_exp-7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

### Visualización de Resultados yolo26m
Visualización del archivo `results.png` generado durante el experimento de hiperparámetros.

![Resultados yolo26m](runs/train/26m/yolo_vertex_exp-7/results.png)

A diferencia del modelo *nano*, el modelo *medium* (`yolo26m`) tiene una mayor capacidad de aprendizaje. Observando las curvas de pérdida y métricas a lo largo de las épocas, esperamos ver una mejor capacidad para extraer características finas y una estabilización superior en el rendimiento de validación.

#### Matriz de Confusión Normalizada
![Matriz de Confusión Normalizada yolo26m](runs/train/26m/yolo_vertex_exp-7/confusion_matrix_normalized.png)

Al observar detalladamente esta matriz de confusión y compararla con el modelo base, esperamos encontrar una diagonal principal con valores más cercanos a 1.0. Gracias a la mayor complejidad topológica de la variante "medium", la incidencia de falsos negativos hacia la clase de "fondo" (Background) debería reducirse de forma evidente, demostrando que al modelo `yolo26m` le cuesta menos distinguir aquellas texturas o variaciones de luz complejas que típicamente confundían al modelo *nano*.

In [ ]:
print(results.results_dict)

{'metrics/precision(B)': 0.9722498514602783, 'metrics/recall(B)': 0.9659626149516796, 'metrics/mAP50(B)': 0.990389205472424, 'metrics/mAP50-95(B)': 0.8594385060204313, 'fitness': 0.8594385060204313}


#### Análisis de Resultados y Métricas (mAP50-95) para yolo26m
Al evaluar los resultados finales del modelo superior (`yolo26m`), confirmamos un rendimiento sobresaliente que justifica plenamente su mayor número de parámetros. Las métricas obtenidas son concluyentes:

- **Precisión (Precision):** 97.2%
- **Recall:** 96.6%
- **mAP50:** 99.0%
- **mAP50-95:** 85.9%

En comparación al modelo *nano*, la configuración *medium* saltó a un **85.9% en el mAP50-95** (el criterio más estricto del proyecto). A su vez, el Recall se incrementó notablemente hasta un 96.6%, indicando que las omisiones de detecciones (falsos negativos) se redujeron drásticamente. 

La arquitectura más profunda nos ha permitido clasificar de forma robusta e identificar objetos con un margen de precisión espacial altísimo, demostrando que en el contexto de este proyecto (donde nos interesa fundamentalmente el rendimiento en el mAP50-95), la variante "m" de YOLO resulta la mejor opción, compensando significativamente su peso computacional extra.

In [ ]:
print(results_test.results_dict)

{'metrics/precision(B)': 0.9700733433827681, 'metrics/recall(B)': 0.9646033546340934, 'metrics/mAP50(B)': 0.9900387600980302, 'metrics/mAP50-95(B)': 0.8598890151877351, 'fitness': 0.8598890151877351}


### Conclusión y Selección Final del Modelo
En base a la comparativa de las métricas obtenidas entre la versión *nano* (`yolo26n`), la versión *small* (`yolo26s`) y la versión *medium* (`yolo26m`), se evidencia una mejora sustancial al utilizar una red de mayor capacidad. El modelo `yolo26m` ha demostrado un desempeño superior, destacando de manera significativa en la métrica principal y más restrictiva estipulada para el éxito de este proyecto (el **mAP50-95** con un asombroso **85.9%**), junto con un nivel muy bajo de falsos negativos.

Por lo tanto, **el modelo definitivo seleccionado para avanzar hacia la siguiente etapa y ser evaluado formalmente sobre el conjunto de imágenes de Test, será el `yolo26m`**.